[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 00](README.md)

# Entorno reproducible y diagnóstico

**Tema:** 00 · **Sesiones:** 1, 2 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo demostrar que una práctica puede construirse y repetirse en el equipo disponible?


## Resultados de aprendizaje

- Distinguir plataforma, toolchain, runtime y dependencia.
- Registrar evidencia mínima del sistema sin confundir disponibilidad con compatibilidad.
- Ejecutar el preflight y leer sus informes antes de iniciar una práctica.


## Modelo conceptual

Un entorno reproducible declara versiones, arquitectura y comandos; no se reduce a una lista de paquetes.

El manifiesto de plataforma describe lo observado. La política del ejercicio determina si ese equipo es compatible.

CMake configura y CTest verifica corrección; las mediciones de rendimiento se realizan solo después de superar las pruebas.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "00"
NOTEBOOK = "00_entorno/00_entorno_reproducible.ipynb"
assert (ROOT / "curso" / "notebooks" / "00_entorno" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Inventario local

Se inspeccionan datos portables del intérprete y la presencia de herramientas sin instalar ni modificar el sistema.


In [ ]:
import platform, shutil, sys
inventory = {
    "python": sys.version.split()[0],
    "os": platform.system(),
    "release": platform.release(),
    "machine": platform.machine(),
    "logical_cpus": __import__("os").cpu_count(),
    "cmake": shutil.which("cmake"),
    "ctest": shutil.which("ctest"),
    "cc": shutil.which("cc"),
    "cxx": shutil.which("c++"),
}
assert inventory["logical_cpus"] and inventory["logical_cpus"] > 0
for key, value in inventory.items():
    print(f"{key:12}: {value}")


**Interpretación.** Una ruta ausente se reporta como evidencia diagnóstica; no se sustituye por una afirmación de soporte.


## Configuración declarada

Se extraen las versiones canónicas del toolchain y se comprueba que las claves esenciales estén presentes.


In [ ]:
import re
toolchain = (ROOT / "config" / "course-toolchain.cmake").read_text(encoding="utf-8")
pairs = dict(re.findall(r'set\((COURSE_[A-Z0-9_]+) "([^"]+)"', toolchain))
required = {"COURSE_GCC_VERSION", "COURSE_CXX_STANDARD", "COURSE_MPI_VERSION", "COURSE_CUDA_VERSION", "COURSE_PYTHON_VERSION"}
assert required <= pairs.keys(), required - pairs.keys()
for key in sorted(required):
    print(f"{key}={pairs[key]}")


**Interpretación.** La versión declarada es un requisito; el manifiesto del equipo permite contrastarla con la versión observada.


## Práctica reproducible

1. Ejecutar `python3 validation/preflight.py` desde la raíz.
2. Conservar los JSON de `build/validation/preflight/`.
3. Explicar qué comprobó cada etapa y qué no demuestra todavía.


## Errores frecuentes

- Continuar aunque el preflight falle.
- Confundir arquitectura con fabricante de CPU.
- Afirmar soporte de GPU porque `nvcc` está instalado sin ejecutar en un dispositivo.

## Criterios de aceptación

- Preflight con código de salida cero.
- Manifiesto de plataforma adjunto al informe.
- Limitaciones del equipo descritas explícitamente.


## Referencias y material relacionado

- [Protocolo de reproducibilidad](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)
- [Configuración del curso](../../../config/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 00](README.md)
